# Arrow IPC - Python

All 11 Python examples from [docs/ipc.md](https://platob.github.io/yggdryl/ipc/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

## Arrow batch reads and writes

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

schema = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("venue", pa.string()),
])
batch = lambda ids, venues: pa.record_batch(
    {"id": ids, "venue": venues}, schema=schema
)

# The name says Arrow IPC, so no call names an encoding.
handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.arrows")
handle.overwrite_arrow_record_batch(batch([1, 2], ["XNAS", "XNYS"]))
handle.append_arrow_record_batch(batch([3], ["XLON"]))

merging = handle.record_options()
merging.merge_by_names = ["id"]
handle.merge_arrow_record_batch(batch([2, 4], ["XPAR", None]), options=merging)

assert handle.read_arrow_field().name == "row"
assert handle.read_arrow_reader().read_all().num_rows == 4

### Dimensions and opened sessions

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "dimensions.arrows")
handle.overwrite_arrow_table(pa.table({"id": [1, 2]}))
with handle:
    assert (handle.row_size, handle.column_size) == (2, 1)
    assert handle.read_arrow_field().name == "row"

## Reading and writing are both readers

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

schema = pa.schema([pa.field("id", pa.int64(), nullable=False)])
batches = [
    pa.record_batch({"id": [start, start + 1]}, schema=schema)
    for start in range(0, 6, 2)
]

handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.arrows")

# The primitive write consumes exactly one RecordBatchReader.
handle.overwrite_arrow_reader(pa.RecordBatchReader.from_batches(schema, batches))

reader = handle.read_arrow_reader()
# The schema is known before a single batch is decoded.
assert reader.schema.names == ["id"]

rows = sum(batch.num_rows for batch in reader)
assert rows == 6

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

schema = pa.schema([pa.field("id", pa.int64(), nullable=False)])
handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.arrows")

# Nothing is materialized: each batch is built as the writer asks for it.
produced = (
    pa.record_batch({"id": [start]}, schema=schema) for start in range(4)
)
handle.overwrite_arrow_reader(pa.RecordBatchReader.from_batches(schema, produced))

assert sum(1 for _ in handle.read_arrow_reader()) == 4

## Column pushdown

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

stored = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("symbol", pa.string(), nullable=False),
    pa.field("venue", pa.string(), nullable=False),
])
batch = pa.record_batch(
    {"id": [1, 2], "symbol": ["AAPL", "MSFT"], "venue": ["XNAS", "XNAS"]},
    schema=stored,
)

handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.arrows")
handle.overwrite_arrow_record_batch(batch)

# One of the three columns, declared through the centralized options field.
options = handle.record_options()
options.field = pa.schema([pa.field("id", pa.int64(), nullable=False)])
projected = handle.read_arrow_reader(options=options)
assert projected.schema.names == ["id"]
assert projected.read_all().num_columns == 1

# The stream itself is unchanged: it still carries all three.
assert len(handle.read_arrow_field().data_type) == 3

## The stream carries its schema

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

schema = pa.schema([pa.field("id", pa.int64(), nullable=False)])
handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.arrows")
handle.overwrite_arrow_record_batch(pa.record_batch({"id": [7]}, schema=schema))

# A reader that declares nothing recovers the schema from the bytes.
assert handle.read_arrow_field().name == "row"

# Arrow names columns, not the record; the root name is chosen on this side.
named = handle.record_options()
named.root_name = "trade"
assert handle.read_arrow_field(options=named).name == "trade"
assert [child.name for child in handle.read_arrow_field().data_type] == ["id"]

## Content coding comes from the name

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

schema = pa.schema([pa.field("id", pa.int64(), nullable=False)])
root = pathlib.Path(tempfile.mkdtemp())

written = []
for name in ("trades.arrows", "trades.arrows.gz", "trades.arrows.zst"):
    handle = IOBase(root / name)
    handle.overwrite_arrow_record_batch(pa.record_batch({"id": [1, 2]}, schema=schema))

    # Identical calls on both sides, whatever the coding is.
    assert handle.read_arrow_reader().read_all().num_rows == 2, name
    written.append(handle.read_bytes())

# The bytes underneath are framed by the coding the name declared.
assert written[1][:2] == bytes.fromhex("1f8b")
assert written[2][:4] == bytes.fromhex("28b52ffd")
assert written[0] != written[1]

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

schema = pa.schema([pa.field("id", pa.int64(), nullable=False)])
handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.arrows.gz")

options = handle.record_options()
options.level = 9
handle.overwrite_arrow_record_batch(
    pa.record_batch({"id": list(range(512))}, schema=schema), options=options
)

assert handle.read_arrow_reader().read_all().num_rows == 512
# Still a gzip member, and smaller than the stream it encodes.
assert handle.read_bytes()[:2] == bytes.fromhex("1f8b")
assert handle.size < 512 * 8

## Options

In [ ]:
import pyarrow as pa

from yggdryl import RecordOptions

schema = pa.schema([pa.field("id", pa.int64(), nullable=False)])

# The media type names the encoding, so there is no format argument.
options = RecordOptions("trades.arrows")
options.field = schema
options.level = 9

assert options.field is not None
assert options.root_name == "row"
assert options.level == 9

options.batch_size = 1024
assert options.batch_size == 1024

assert str(options.mime_type) == "application/vnd.apache.arrow.stream"
# A setting another encoding has is absent rather than invented here.
assert options.max_row_group_size is None

## Absence

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

schema = pa.schema([pa.field("id", pa.int64(), nullable=False)])
root = pathlib.Path(tempfile.mkdtemp())

# A resource that does not exist yet holds no batches; it is not a parse failure.
missing = IOBase(root / "missing.arrows")
assert not missing.exists()
assert missing.read_arrow_reader().read_all().num_rows == 0

# Writing no batches still writes the schema, so the stream exists and reads.
written = IOBase(root / "empty.arrows")
written.overwrite_arrow_table(pa.Table.from_batches([], schema=schema))
assert written.size > 0
assert written.read_arrow_reader().read_all().num_rows == 0
assert written.read_arrow_field().name == "row"

In [ ]:
import pytest

from yggdryl import IOBase

handle = IOBase.from_bytes(b"definitely not an Arrow IPC stream")
handle.media_type = "application/vnd.apache.arrow.stream"

with pytest.raises(ValueError):
    handle.read_arrow_field()
with pytest.raises(ValueError):
    handle.read_arrow_reader()